# Handout – Block 9: Templates, Bedingungen und Handler

Dieses Handout erläutert die Lerninhalte des Blocks in verständlicher Form. Ausführbare Beispiele stehen in eigenen Codezellen. Beispiele, die eine vorbereitete Ansible-Trainingsumgebung oder administrative Rechte benötigen, sind so gekennzeichnet, dass sie nicht versehentlich auf einem beliebigen Notebook-System ausgeführt werden.

> Das Handout ergänzt Trainerleitfaden und Übungsnotebook: Es erklärt die Konzepte; die umfangreicheren Aufgaben bleiben im Übungsnotebook.

## 1. Warum Templates?

Eine statische Datei ist für alle Hosts identisch. Ein **Jinja2-Template** kann aus Variablen und Facts hostabhängige Konfiguration erzeugen.

Template:

```jinja2
server_name = {{ inventory_hostname }}
port = {{ web_port }}
```

Ausgabe für `web01` kann dadurch andere Werte enthalten als für `web02`.

## 2. Template-Modul

```yaml
- name: Konfiguration erzeugen
  ansible.builtin.template:
    src: app.conf.j2
    dest: /etc/app.conf
```

Ansible rendert das Template auf dem Control Node mit den für den Zielhost geltenden Variablen und überträgt das Ergebnis.

## 3. Bedingungen

Tasks können mit `when` abhängig von Variablen oder Facts ausgeführt werden:

```yaml
when: ansible_facts['os_family'] == 'Debian'
```

Auch Jinja2 kennt Bedingungen:

```jinja2
{% if tls_enabled %}
ssl = on
{% endif %}
```

## 4. Schleifen

Playbook:

```yaml
- name: Verzeichnisse anlegen
  ansible.builtin.file:
    path: "{{ item }}"
    state: directory
  loop:
    - /opt/app
    - /opt/app/data
```

Template:

```jinja2
{% for host in backend_hosts %}
server {{ host }};
{% endfor %}
```

## 5. Handler und `notify`

Ein Dienst soll nur dann neu geladen werden, wenn sich seine Konfiguration tatsächlich geändert hat.

```text
template
   │
   ├── ok ───────────────► nichts tun
   │
   └── changed
          │ notify
          ▼
       Handler
          │
          ▼
      reload/restart
```

Das verbindet Änderungserkennung und Idempotenz.

Beispiel:

```yaml
- name: Konfiguration erzeugen
  ansible.builtin.template:
    src: app.conf.j2
    dest: /etc/app.conf
  notify: App neu laden

handlers:
  - name: App neu laden
    ansible.builtin.service:
      name: app
      state: reloaded
```

### Selbstkontrolle

1. Warum sind Templates gegenüber statischen Dateien flexibler?
2. Wofür steht `{{ ... }}`?
3. Wozu dient `when`?
4. Wo können Schleifen eingesetzt werden?
5. Wann wird ein Handler durch `notify` vorgemerkt?
6. Wie unterstützen Handler die Idempotenz?

# Ausblick / Abschluss von Block 9

Die Konzepte dieses Blocks bilden die Grundlage für den folgenden Seminarabschnitt. Nutzen Sie das separate Übungsnotebook, um die hier erläuterten Inhalte praktisch zu vertiefen.